# 🐍 Day 08 · Python Basics — String Methods & Smarter Chunking

> **Goal of this notebook:** How you cut a document decides how good RAG answers are. Learn string tools and build fixed-size, overlapping and recursive chunkers yourself.

**How to use it**
1. Click on a grey code cell and press **Shift + Enter** to run it.
2. Read the comments (lines that start with `#`) — they explain every step.
3. Change the values and run the cell again. Breaking things is how you learn! 🙂

📦 **First** do *Step 1 - Install today's packages* in [`00_START_HERE.md`](00_START_HERE.md).

✅ No API key needed for this notebook.

### 📋 What you will learn
1. Useful string methods
2. Character chunks with overlap
3. `enumerate()` to number things
4. Recursive splitting (paragraph → sentence → word)
5. Adding metadata to chunks
6. Exporting chunks to JSON

## 1. Useful string methods

In [ ]:
text = "   Attention Is All You Need.\nTransformers use attention.   "

print(text.strip())                       # remove spaces/newlines at both ends
print(text.lower().count("attention"))    # how many times a word appears
print(text.find("Transformers"))          # position of a word (-1 if missing)
print(text.strip().startswith("Attention"))
print(text.strip().splitlines())          # split at line breaks
print(text.replace("Need", "Want"))

## 2. Character chunks with **overlap**
If we cut exactly at the edge, a sentence can be split in half. **Overlap** repeats a few characters so no meaning is lost.
```
chunk 1: [0 ............ 40]
chunk 2:           [30 ............ 70]   <- 10 characters overlap
```

In [ ]:
def chunk_chars(text, chunk_size=40, overlap=10):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap          # step back a little = overlap
    return chunks

sample = "Transformers read the whole sentence at once. Attention tells the model which words matter most."
for c in chunk_chars(sample):
    print(repr(c))                     # repr shows the exact text including spaces

## 3. `enumerate()` — get the number AND the item

In [ ]:
fruits = ["apple", "banana", "cherry"]
for number, fruit in enumerate(fruits, start=1):
    print(number, fruit)

## 4. Recursive splitting
`RecursiveCharacterTextSplitter` tries the **biggest** separator first (paragraphs). If a piece is still too long, it tries the next one (sentences), then words.
A **recursive** function is a function that calls itself.

In [ ]:
def recursive_split(text, max_len=60, separators=["\n\n", ". ", " "]):
    if len(text) <= max_len or not separators:   # small enough -> done
        return [text]
    sep = separators[0]                           # try the biggest separator
    pieces = []
    for part in text.split(sep):
        # if a part is still too long, split it again with the NEXT separators
        pieces.extend(recursive_split(part, max_len, separators[1:]))
    return [p for p in pieces if p.strip()]       # drop empty pieces

doc = ("RAG has two parts.\n\nFirst we retrieve useful chunks from a vector database. "
       "Then the LLM writes an answer using only those chunks.")
for i, piece in enumerate(recursive_split(doc), start=1):
    print(i, "|", piece)

## 5. Metadata — remember where each chunk came from
Real RAG systems store extra info (file, page, chunk number) next to every chunk, so the answer can show its **source**.

In [ ]:
chunks = []
for i, piece in enumerate(recursive_split(doc)):
    chunks.append({
        "chunk_id": i,
        "source": "rag_notes.txt",
        "page": 1,
        "text": piece,
        "length": len(piece),
    })

for c in chunks:
    print(c["chunk_id"], c["length"], c["text"][:40])

## 6. Export chunks to JSON
`8_multi_modal_rag.ipynb` saves its chunks to `chunks_export.json` in the same way.

In [ ]:
import json, os

with open("demo_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2)

print(open("demo_chunks.json", encoding="utf-8").read()[:200], "...")
os.remove("demo_chunks.json")      # tidy up

---
## 🏋️ Practice time
Try these on your own in the empty cell below:

1. Count how many times the letter `a` appears in `"banana bandana"`.
2. Run `chunk_chars(sample, chunk_size=30, overlap=5)` and count the chunks.
3. Use `enumerate` to print `1. GPT`, `2. BERT`, `3. T5`.

In [ ]:
# ✍️ Write your answers here

<details>
<summary>👀 Click here to see the solutions (try first!)</summary>

```python
print("banana bandana".count("a"))   # 6

print(len(chunk_chars(sample, chunk_size=30, overlap=5)))

for i, m in enumerate(["GPT", "BERT", "T5"], start=1):
    print(f"{i}. {m}")
```
</details>

## ✅ Summary

| Concept | Example |
|---|---|
| Clean text | `text.strip().lower()` |
| Find / count | `text.find('x'), text.count('x')` |
| Overlap chunks | `start = end - overlap` |
| Number items | `for i, x in enumerate(items, start=1)` |
| Recursion | `a function that calls itself` |
| Add many items | `pieces.extend(other_list)` |
| Metadata | `{'source': 'a.txt', 'page': 1, 'text': t}` |

## ➡️ What's next?
You now know the Python used in **Day_08_RAG_Chunking_Strategies**.

Open [`00_START_HERE.md`](00_START_HERE.md) and continue with **lesson 1**. 🚀